# Tarea 3 · Parte 1: Obtención y análisis de datos

## 1.1. Definición del problema

- *Elegir un tema y formular una pregunta concreta.*
- *Identificar la cobertura geográfica y temporal del análisis.*

En el trabajo de suficiencia profesional de la autora (2025), sobre la operatividad del Sistema de Focalización de Hogares (SISFOH), se describió el rol clave que cumplen las municipalidades para la determinación de la Clasificación Socioeconómica (CSE) de los hogares. 

Las municipalidades, a través de las Unidades Locales de Empadronamiento (ULE), son las responsables de recepcionar la solicitud de determinación de la CSE, recolectar la información del hogar a través del Formato S100 y la Ficha Socioeconómica Única (FSU), y digitarla en los aplicativos proporcionados por el MIDIS-OFIS. Además, el alcalde debe firmar las Declaraciones Juradas (DJ), mediante las cuales manifiesta que el levantamiento de información y la digitación se realizaron de acuerdo con los lineamientos establecidos en la directiva vigente sobre la operatividad del SISFOH. Cuando dichas DJ son procesadas a través de los aplicativos del MIDIS-OFIS, automáticamente se calcula la CSE del hogar, de acuerdo con el algoritmo de la metodología vigente.

Frente a lo expuesto, se puede afirmar lo siguiente:

- Las ULE son el brazo operativo del SISFOH.
- La calidad de los datos recolectados es fundamental en la determinación de la CSE, ya que es un algoritmo el que calcula la CSE. Un error al definir quiénes son los integrantes del hogar, o digitar mal el número de suministro de luz de la vivienda, u omitir que un integrante del hogar tiene alguna discapacidad, puede generar una CSE que no corresponde a la realidad del hogar. Por ello, es necesario que el equipo técnico de la ULE se encuentre capacitado, tenga experiencia y los recursos para poder desempeñar sus labores de acuerdo con lo indicado en la directiva vigente sobre la operatividad del SISFOH.

Ahora, el Registro Nacional de Municipalidades (RENAMU) del INEI recolecta, anualmente, información integral de las municipalidades provinciales y distritales; entre otros aspectos, los recursos con los que cuentan, la cantidad de personal y su tipo de contrato.

Inicialmente, se planteó analizar si existe relación entre los recursos de las municipalidades y la proporción de hogares empadronados por distrito. Sin embargo, esa información no es pública a nivel distrital: obtenerla requiere una solicitud al OFIS, que tiene un plazo de 10 días hábiles para atenderla, lo cual excede el plazo de esta tarea. No obstante, este ejercicio permite un primer acercamiento a la relación entre las capacidades de las municipalidades y su necesidad de capacitación y asistencia técnica sobre la operatividad del SISFOH.

Para ello, en esta tarea se usan solo datos del RENAMU sobre los recursos materiales y humanos de las municipalidades, y se analiza su relación con las únicas variables del registro vinculadas al SISFOH: si la municipalidad declara que requiere capacitación o asistencia técnica. Se espera que las municipalidades con menos capacidades (sin conexión fija a internet, con pocas computadoras o con personal mayoritariamente temporal) declaren con más frecuencia que requieren capacitación o asistencia técnica sobre la operatividad del SISFOH.

La pregunta de investigación es:

> **¿Las municipalidades con menos capacidades (conectividad, equipos, personal estable) son las que más requieren capacitación y asistencia técnica sobre la operatividad del SISFOH? Perú, 2024–2026.**

**Cobertura geográfica:** municipalidades provinciales y distritales del Perú: las mismas 1 891 en cada edición (196 provinciales y 1 695 distritales). Se excluye Santa Rosa de Loreto, distrito que solo aparece en la edición 2026.


**Cobertura temporal:** ediciones 2024, 2025 y 2026 del RENAMU.

## 1.2. Obtención de datos

- *Descargar o recopilar información de una o varias fuentes verificables.*
- *Documentar el origen y la fecha de los datos.*
- *Guardar los archivos utilizados o registrar cómo obtenerlos.*

**Fuente:** Registro Nacional de Municipalidades (RENAMU), Formulario 01, ediciones 2024, 2025 y 2026. Instituto Nacional de Estadística e Informática (INEI).

**Cómo obtener los archivos:**

1. Ingresar al [portal de microdatos del INEI](https://proyectos.inei.gob.pe/microdatos/), sección *Consulta por encuestas*, y elegir RENAMU y el año.
2. Descargar la base del Formulario 01 en formato STATA (`.dta`) y el diccionario de variables.
3. Renombrar cada base y guardarla en `datos/originales/`:

    - 2024: `Base-Datos_2024_f.dta` renombrarlo a `renamu_2024.dta`
    - 2025: `Base-Datos_2025_f_.dta` renombrarlo a `renamu_2025.dta`
    - 2026: `Base_de_Datos_F01_RENAMU2026.dta` renombrarlo a `renamu_2026.dta`

**Fecha de descarga:** 27 de octubre de 2026.

Las bases originales no se publican en el repositorio (están excluidas en `.gitignore`) porque incluyen datos de contacto de los alcaldes. Los diccionarios de variables se guardan en `datos/diccionarios/`.

En este notebook se cargan las tres bases, se seleccionan las variables de interés, se revisa la calidad de los datos, se construyen los indicadores y se guarda una base limpia en `datos/procesados/`.


In [1]:
# Librerías
from pathlib import Path
import pandas as pd

# Carpetas del proyecto
CARPETA_ORIGINALES = Path("datos/originales")
CARPETA_PROCESADOS = Path("datos/procesados")

# Archivo de cada año
ARCHIVOS = {
    2024: CARPETA_ORIGINALES / "renamu_2024.dta",
    2025: CARPETA_ORIGINALES / "renamu_2025.dta",
    2026: CARPETA_ORIGINALES / "renamu_2026.dta",
}

# Se comprueba que cada archivo existe
for anio, ruta in ARCHIVOS.items():
    print(anio, ruta, ruta.exists())

2024 datos\originales\renamu_2024.dta True
2025 datos\originales\renamu_2025.dta True
2026 datos\originales\renamu_2026.dta True


### Selección de variables

Cada base tiene cerca de 1 390 columnas. Se conservan solo las necesarias para responder la pregunta, agrupadas en seis dimensiones:

1. **Identificación:** `Ubigeo`, `Departamento`, `Provincia`, `Distrito` y `Tipomuni` (datos generales).
2. **Capacitación y asistencia técnica sobre la operatividad del SISFOH (resultado):** `P22_AT23` y `P22_C23`. Pregunta 22: asistencia técnica y capacitación que requiere la municipalidad en el Sistema de Focalización de Hogares.
3. **Conectividad:** `P14`, `P14A_1` y `P14A_2`. Pregunta 14: acceso a internet, computadoras con internet y tipo de conexión.
4. **Equipos:** `P13A_1` a `P13A_10`. Pregunta 13: computadoras operativas por tipo de procesador.
5. **Personal:** `P19D_T` y las variables `P19D_*` por régimen laboral. Pregunta 19: personal al 31 de diciembre del año anterior.
6. **Contexto:** `P60B_T_1` (beneficiarios del Vaso de Leche) y `P101_4` (participación en la Mesa de Concertación para la Lucha contra la Pobreza).

**Nota:** el total de computadoras (`P13A_T`) no existe en la edición 2024; por eso se suman las computadoras por tipo de procesador.


In [2]:
# Identificación de la municipalidad
ID = ["Ubigeo", "Departamento", "Provincia", "Distrito", "Tipomuni"]

# Resultado: ¿requiere asistencia técnica o capacitación en el SISFOH?
SISFOH = ["P22_AT23", "P22_C23"]

# Conectividad
CONECTIVIDAD = ["P14", "P14A_1", "P14A_2"]

# Equipos: computadoras por tipo de procesador (P13A_1 a P13A_10)
COMPUTADORAS = [f"P13A_{i}" for i in range(1, 11)]

# Personal por régimen laboral (N = 276 nombrado, C = 276 contratado,
# L = 728, V = CAS; M = mujer, H = hombre)
PERSONAL = ["P19D_T", "P19D_NM", "P19D_NH", "P19D_CM", "P19D_CH",
            "P19D_LM", "P19D_LH", "P19D_VM", "P19D_VH"]

# Contexto
CONTEXTO = ["P60B_T_1", "P101_4"]

COLUMNAS = ID + SISFOH + CONECTIVIDAD + COMPUTADORAS + PERSONAL + CONTEXTO
print(len(COLUMNAS), "columnas seleccionadas")


31 columnas seleccionadas


### Carga de las bases

Luego, se lee cada archivo `.dta` (formato de Stata), se agrega una columna con el año de la edición y se apilan las tres tablas en una sola.

In [3]:
tablas = []  # lista vacía donde se guarda la tabla de cada año

for anio, ruta in ARCHIVOS.items():
    tabla = pd.read_stata(ruta, columns=COLUMNAS, convert_categoricals=False)
    tabla["anio"] = anio
    tablas.append(tabla)
    print(anio, tabla.shape)

renamu = pd.concat(tablas, ignore_index=True)
renamu.shape


2024 (1891, 32)
2025 (1891, 32)
2026 (1892, 32)


(5674, 32)

In [4]:
# Visualización de los primeros registros
renamu.head()

,Ubigeo,Departamento,Provincia,Distrito,Tipomuni,P22_AT23,P22_C23,P14,P14A_1,P14A_2,...,P19D_NH,P19D_CM,P19D_CH,P19D_LM,P19D_LH,P19D_VM,P19D_VH,P60B_T_1,P101_4,anio
0,010101,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,1.0,23.0,0.0,1.0,135.0,4.0,...,14.0,3.0,7.0,4.0,14.0,86.0,120.0,780.0,4.0,2024
1,010102,AMAZONAS,CHACHAPOYAS,ASUNCION,2.0,23.0,23.0,1.0,4.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.0,0.0,2024
2,010103,AMAZONAS,CHACHAPOYAS,BALSAS,2.0,0.0,0.0,1.0,6.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,205.0,0.0,2024
3,010104,AMAZONAS,CHACHAPOYAS,CHETO,2.0,23.0,23.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0,0.0,2024
4,010105,AMAZONAS,CHACHAPOYAS,CHILIQUIN,2.0,23.0,23.0,1.0,4.0,1.0,...,1.0,0.0,0.0,0.0,0.0,1.0,1.0,68.0,0.0,2024


## 1.3. Procesamiento
- *Limpiar registros inconsistentes o duplicados.*
- *Transformar variables cuando sea necesario.*
- *Construir indicadores adecuados para responder la pregunta.*

### Revisión de calidad

Antes de construir los indicadores, se revisa que:

1. No haya municipalidades repetidas en un mismo año;
2. El número de municipalidades por tipo sea coherente entre los tres años;
3. Las variables de las preguntas estén en formato numérico;
4. Se identifiquen los valores faltantes y lo que significan.

In [5]:
# 1. ¿Hay municipalidades repetidas en un mismo año?
duplicados = renamu.duplicated(subset=["Ubigeo", "anio"])
print("Registros duplicados:", duplicados.sum())

Registros duplicados: 0


In [6]:
# 2. Número de municipalidades por año y tipo (1 = provincial, 2 = distrital)
pd.crosstab(renamu["anio"], renamu["Tipomuni"])

Tipomuni,1.0,2.0
anio,,
2024,196,1695
2025,196,1695
2026,196,1696


In [7]:
# 3. Tipo de dato de cada columna
renamu.dtypes

Ubigeo              str
Departamento        str
Provincia           str
Distrito            str
Tipomuni        float64
P22_AT23        float64
P22_C23         float64
P14             float64
P14A_1          float64
P14A_2           object
P13A_1          float64
P13A_2          float64
P13A_3          float64
P13A_4          float64
P13A_5          float64
P13A_6          float64
P13A_7          float64
P13A_8          float64
P13A_9          float64
P13A_10         float64
P19D_T          float64
P19D_NM         float64
P19D_NH         float64
P19D_CM         float64
P19D_CH         float64
P19D_LM         float64
P19D_LH         float64
P19D_VM         float64
P19D_VH         float64
P60B_T_1        float64
P101_4          float64
anio              int64
dtype: object

In [8]:
# Variables que deben ser numéricas: todas menos las de identificación
NUMERICAS = SISFOH + CONECTIVIDAD + COMPUTADORAS + PERSONAL + CONTEXTO

# Se convierten a número; si un valor no se puede convertir, queda vacío (NaN)
renamu[NUMERICAS] = renamu[NUMERICAS].apply(pd.to_numeric, errors="coerce")

renamu["P14A_2"].dtype

dtype('float64')

In [9]:
# 4. Valores faltantes (NaN) por variable y año
faltantes = renamu[NUMERICAS].isna().groupby(renamu["anio"]).sum()
faltantes.T

anio,2024,2025,2026
P22_AT23,0,0,0
P22_C23,0,0,0
P14,0,0,0
P14A_1,49,45,27
P14A_2,49,45,0
P13A_1,1432,1464,1542
P13A_2,1558,1589,1646
P13A_3,1338,1420,1489
P13A_4,1621,1657,1690
P13A_5,548,573,560


In [10]:
# Tipo de conexión a internet por año, incluidos los vacíos
pd.crosstab(renamu["anio"], renamu["P14A_2"], dropna=False)

P14A_2,0.0,1.0,2.0,3.0,4.0,5.0,NaN
anio,,,,,,,
2024,0,404,132,130,300,876,49
2025,0,391,143,97,320,895,45
2026,27,427,130,75,297,936,0


**Resultados de la revisión:**

- No hay municipalidades duplicadas en un mismo año.
- Cada año hay 196 municipalidades provinciales. En 2026 aparece una distrital más, Santa Rosa de Loreto (ubigeo 160405), que se registra por primera vez en esa edición; se excluye del análisis (ver la sección siguiente).
- `P14A_2` (tipo de conexión) venía como texto en 2026; se convirtió a número junto con el resto de variables.

**Significado de los valores faltantes:**

- **P22_AT23, P22_C23, P14, P60B_T_1 y P101_4** no tienen valores faltantes.
- **P14A_1 y P14A_2** solo están vacías en las municipalidades sin internet (`P14 = 2`): 49 en 2024, 45 en 2025 y 27 en 2026. En 2026, el tipo de conexión de esas municipalidades se registró como 0 y no como vacío, pero significa lo mismo: "no aplica".
- **P13A_1 a P13A_10** tienen muchos vacíos porque cada municipalidad solo registra los tipos de procesador que tiene: un vacío equivale a 0 computadoras de ese tipo. Ninguna municipalidad tiene todos los tipos vacíos.
- **P19D_*** (personal): 308 registros de municipalidades distritales (110 en 2024, 78 en 2025 y 120 en 2026) no tienen información de personal, ni siquiera el total. En este caso el vacío **no** equivale a 0 trabajadores, sino a "sin información"; por eso esos registros se excluirán de los indicadores de personal.


### Exclusión de registros

Se excluye a Santa Rosa de Loreto (ubigeo 160405), distrito que solo aparece en la edición 2026 y cuya operatividad del SISFOH está a cargo de la Municipalidad Distrital de Yavarí, según una [nota de prensa del OFIS](https://www.gob.pe/institucion/ofis/noticias/1227157-el-ofis-y-municipalidades-de-mariscal-ramon-castilla-y-yavari-impulsan-empadronamiento-masivo-en-santa-rosa-de-loreto). Así, en los tres años se comparan las mismas 1 891 municipalidades (196 provinciales y 1 695 distritales).

Santa Rosa de Loreto tampoco tenía información de personal; por eso, tras la exclusión, en 2026 quedan 119 registros sin esa información (307 en total)

In [11]:
# Se excluye Santa Rosa de Loreto (solo aparece en 2026)
renamu = renamu[renamu["Ubigeo"] != "160405"]
print(renamu.shape)

# Verificación: cada año debe tener 196 provinciales y 1 695 distritales
pd.crosstab(renamu["anio"], renamu["Tipomuni"])

(5673, 32)


Tipomuni,1.0,2.0
anio,,
2024,196,1695
2025,196,1695
2026,196,1695


### Construcción de indicadores

| Indicador | Definición |
|---|---|
| `requiere_asistencia` | 1 si requiere asistencia técnica sobre el SISFOH (`P22_AT23 = 23`); 0 si no |
| `requiere_capacitacion` | 1 si requiere capacitación sobre el SISFOH (`P22_C23 = 23`); 0 si no |
| `requiere_alguna` | 1 si requiere asistencia técnica o capacitación |
| `tiene_internet` | 1 si cuenta con servicio de internet (`P14 = 1`) |
| `conexion_fija` | 1 si la conexión que más usa es Wi-Fi, ADSL/DSL o fibra óptica (`P14A_2` = 1, 3 o 5) |
| `computadoras` | Total de computadoras operativas (suma de `P13A_1` a `P13A_10`) |
| `personal_total` | Personal al 31 de diciembre del año anterior (`P19D_T`) |
| `pct_personal_estable` | % de personal nombrado (D. Leg. 276) o del régimen del D. Leg. 728 |
| `comp_por_trabajador` | Computadoras por trabajador |

**Nota sobre el personal estable:** no se considera estable al personal contratado del D. Leg. 276 ni al personal CAS. Si bien desde la Ley N.° 31131 (2021) los contratos CAS que cubren funciones permanentes son a plazo indeterminado, el RENAMU no distingue entre CAS indeterminados y temporales, y en la práctica la aplicación de esa norma no ha sido uniforme entre municipalidades. Por ello, el indicador puede subestimar la proporción de personal estable.

In [12]:
# En la pregunta 22, el código 23 indica que la municipalidad requiere apoyo en el SISFOH
renamu["requiere_asistencia"] = (renamu["P22_AT23"] == 23).astype(int)
renamu["requiere_capacitacion"] = (renamu["P22_C23"] == 23).astype(int)

# Requiere al menos uno de los dos
renamu["requiere_alguna"] = (
    (renamu["requiere_asistencia"] == 1) | (renamu["requiere_capacitacion"] == 1)
).astype(int)

# Porcentaje de municipalidades por año
(renamu.groupby("anio")[["requiere_asistencia", "requiere_capacitacion", "requiere_alguna"]].mean() * 100).round(1)

,requiere_asistencia,requiere_capacitacion,requiere_alguna
anio,,,
2024,59.5,71.8,79.2
2025,60.4,72.9,80.4
2026,60.0,68.3,77.0


In [13]:
# Conectividad
renamu["tiene_internet"] = (renamu["P14"] == 1).astype(int)

# Conexión fija: Wi-Fi (1), ADSL/DSL (3) o fibra óptica (5).
# No se consideran la banda ancha móvil (2) ni la satelital (4).
renamu["conexion_fija"] = renamu["P14A_2"].isin([1, 3, 5]).astype(int)

# Equipos: total de computadoras de todos los tipos de procesador
renamu["computadoras"] = renamu[COMPUTADORAS].sum(axis=1)

(renamu.groupby("anio")[["tiene_internet", "conexion_fija"]].mean() * 100).round(1)

,tiene_internet,conexion_fija
anio,,
2024,97.4,74.6
2025,97.6,73.1
2026,98.6,76.0


In [14]:
# Personal al 31 de diciembre del año anterior
renamu["personal_total"] = renamu["P19D_T"]

# Personal estable: nombrados del D. Leg. 276 y trabajadores del D. Leg. 728 (mujeres y hombres)
renamu["personal_estable"] = (
    renamu["P19D_NM"] + renamu["P19D_NH"] + renamu["P19D_LM"] + renamu["P19D_LH"]
)

renamu["pct_personal_estable"] = renamu["personal_estable"] / renamu["personal_total"] * 100
renamu["comp_por_trabajador"] = renamu["computadoras"] / renamu["personal_total"]

renamu[["computadoras", "personal_total", "pct_personal_estable", "comp_por_trabajador"]].describe().round(2)

,computadoras,personal_total,pct_personal_estable,comp_por_trabajador
count,5673.00,5366.00,5366.00,5366.00
mean,56.80,109.31,35.59,1.38
std,123.12,298.26,27.97,1.65
min,1.00,1.00,0.00,0.05
25%,9.00,8.00,11.11,0.54
50%,18.00,21.00,33.33,0.89
75%,50.00,74.00,57.14,1.53
max,2700.00,5485.00,100.00,31.00


### Índice de capacidad municipal

La capacidad de cada municipalidad se resume en un índice de 0 a 3, que suma tres componentes (1 = cumple; 0 = no cumple):

1. **Conectividad:** cuenta con conexión fija a internet.
2. **Equipos:** tiene al menos 0,5 computadoras por trabajador (una por cada dos trabajadores).
3. **Personal:** al menos el 50 % de su personal es estable.

Según el puntaje, la capacidad se clasifica como **Baja** (0–1), **Media** (2) o **Alta** (3). Las municipalidades sin información de personal no reciben índice.

In [15]:
# Municipalidades con información de personal
tiene_personal = renamu["personal_total"].notna()

# Componentes del índice (1 = cumple, 0 = no cumple; vacío si no hay información de personal)
renamu["cap_conexion"] = renamu["conexion_fija"]
renamu["cap_equipos"] = (renamu["comp_por_trabajador"] >= 0.5).astype(int).where(tiene_personal)
renamu["cap_personal"] = (renamu["pct_personal_estable"] >= 50).astype(int).where(tiene_personal)

# Índice: suma de los tres componentes (queda vacío si falta alguno)
COMPONENTES = ["cap_conexion", "cap_equipos", "cap_personal"]
renamu["indice_capacidad"] = renamu[COMPONENTES].sum(axis=1, min_count=3)

# Nivel de capacidad
NIVELES = {0: "Baja", 1: "Baja", 2: "Media", 3: "Alta"}
renamu["nivel_capacidad"] = renamu["indice_capacidad"].map(NIVELES)

pd.crosstab(renamu["anio"], renamu["nivel_capacidad"], dropna=False)

nivel_capacidad,Alta,Baja,Media,NaN
anio,,,,
2024,339,464,978,110
2025,314,508,991,78
2026,356,517,899,119


### Variables de contexto

Se agregan variables que se usarán en el análisis y en el dashboard:

- `tipo_municipalidad`: "Provincial" o "Distrital" (a partir de `Tipomuni`).
- `beneficiarios_vaso_leche`: número de beneficiarios del Programa del Vaso de Leche (`P60B_T_1`).
- `participa_mclcp`: 1 si la municipalidad participa en la Mesa de Concertación para la Lucha contra la Pobreza (`P101_4 = 4`); 0 si no.

In [16]:
# Tipo de municipalidad con etiqueta
renamu["tipo_municipalidad"] = renamu["Tipomuni"].map({1: "Provincial", 2: "Distrital"})

# Contexto
renamu["beneficiarios_vaso_leche"] = renamu["P60B_T_1"]
renamu["participa_mclcp"] = (renamu["P101_4"] == 4).astype(int)

(renamu.groupby("anio")["participa_mclcp"].mean() * 100).round(1)

anio
2024    28.3
2025    29.0
2026    25.4
Name: participa_mclcp, dtype: float64

### Base procesada

Se guarda una base con una fila por municipalidad y año, que contiene las variables de identificación y los indicadores. No incluye datos personales: los nombres, teléfonos y correos de los alcaldes no se seleccionaron al cargar las bases.

In [17]:
# Columnas de la base final: identificación, resultado, capacidades, índice y contexto
COLUMNAS_FINALES = [
    "anio", "Ubigeo", "Departamento", "Provincia", "Distrito", "tipo_municipalidad",
    "requiere_asistencia", "requiere_capacitacion", "requiere_alguna",
    "tiene_internet", "conexion_fija", "computadoras",
    "personal_total", "pct_personal_estable", "comp_por_trabajador",
    "cap_conexion", "cap_equipos", "cap_personal", "indice_capacidad", "nivel_capacidad",
    "beneficiarios_vaso_leche", "participa_mclcp",
]
base_final = renamu[COLUMNAS_FINALES]

# Se guarda en datos/procesados/
ARCHIVO_FINAL = CARPETA_PROCESADOS / "renamu_indicadores_2024_2026.csv"
base_final.to_csv(ARCHIVO_FINAL, index=False, encoding="utf-8-sig")
print(ARCHIVO_FINAL, base_final.shape)

datos\procesados\renamu_indicadores_2024_2026.csv (5673, 22)


In [18]:
# Verificación: se lee el archivo guardado
prueba = pd.read_csv(ARCHIVO_FINAL, dtype={"Ubigeo": str})
print(prueba.shape)
prueba.head(3)

(5673, 22)


,anio,Ubigeo,Departamento,Provincia,Distrito,tipo_municipalidad,requiere_asistencia,requiere_capacitacion,requiere_alguna,tiene_internet,...,personal_total,pct_personal_estable,comp_por_trabajador,cap_conexion,cap_equipos,cap_personal,indice_capacidad,nivel_capacidad,beneficiarios_vaso_leche,participa_mclcp
0,2024,010101,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,Provincial,1,0,1,1,...,268.0,19.402985,0.503731,0,1.0,0.0,1.0,Baja,780.0,1
1,2024,010102,AMAZONAS,CHACHAPOYAS,ASUNCION,Distrital,1,1,1,1,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,20.0,0
2,2024,010103,AMAZONAS,CHACHAPOYAS,BALSAS,Distrital,0,0,0,1,...,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,205.0,0


## 1.4. Análisis

- *Presentar al menos 3 hallazgos relevantes.*
- *Sustentar los resultados con cifras o visualizaciones.*
- *Identificar posibles limitaciones en los datos y su interpretación.*
